# Method B — recover evaluation and finish the 1× results

Your four 1× continuation models have finished training. The 0.5B seed-123 parent GSM prediction
file contains 1,000 records for 500 question IDs; 452 duplicated IDs have differing records.
This recovery **does not pick one version or remove duplicate rows**. It archives the entire
unvalidated file with its metadata, then regenerates that evaluation using the same pinned evaluator.

Use **Python 3 in one active A100 GPU Jupyter session**. Run the cells below one at a time with
**Shift + Enter**, waiting for each to finish. Use this recovery notebook for the remaining evaluation;
leave the original evaluation cell idle. A check blocks recovery if another project evaluation worker
is active, and a lock prevents two recovery launches from running together.

- Existing datasets, trained adapters, training records, runtime source, and original notebook helper stay unchanged.
- Completed predictions with matching validation records and runtime settings are reused.
- Invalid, partial, or unvalidated prediction bundles are archived whole before regeneration.
- The wrapper permits evaluation, summary, and export; it cannot launch training.
- Results remain preliminary while the common parent/development protection checks are unresolved.

If the session expires, rerun this notebook in another A100 session. Completed matching predictions
are reused; an unfinished output is preserved and regenerated. The remaining profiles still need
their evaluations, so this can take more than the single failed 500-question run.


## 1. Install the separate recovery helper

This verifies the original helper matches the reviewed version and creates an evaluation-only copy.
It preserves the original `run_method_b.py`, the original notebook, and all model weights.


In [ ]:
PAYLOAD = {'evaluation_recovery.py': '"""Preserve invalid evaluation files and prevent overlapping recovery launches.\n\nNo rows are deduplicated or selected by correctness. Invalid/unvalidated outputs\nare archived whole, and the unchanged evaluator regenerates that output.\n"""\nfrom collections import Counter, defaultdict\nfrom contextlib import contextmanager\nfrom datetime import datetime, timezone\nimport fcntl\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport uuid\n\nSTUDY = \'B_method_B_document_v3\'\n\n\ndef sha(path):\n    h=hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for chunk in iter(lambda:f.read(1024*1024),b\'\'):h.update(chunk)\n    return h.hexdigest()\n\n\ndef inspect_predictions(path):\n    path=Path(path)\n    if not path.is_file():return {\'exists\':False,\'path\':str(path)}\n    counts=Counter();variants=defaultdict(set);invalid=[];ids=[]\n    differing_fields=Counter();by_id={}\n    for line_no,line in enumerate(path.read_text().splitlines(),1):\n        if not line.strip():continue\n        try:\n            record=json.loads(line)\n            identity=record[\'problem_id\']\n            hash(identity)\n        except (ValueError,KeyError,TypeError) as error:\n            invalid.append({\'line\':line_no,\'error\':type(error).__name__});continue\n        counts[identity]+=1;ids.append(identity)\n        variants[identity].add(json.dumps(record,sort_keys=True,ensure_ascii=False))\n        if identity in by_id:\n            old=by_id[identity]\n            for key in set(old)|set(record):\n                if old.get(key)!=record.get(key):differing_fields[key]+=1\n        else:by_id[identity]=record\n    halfway=len(ids)//2\n    return {\'exists\':True,\'path\':str(path),\'sha256\':sha(path),\'valid_json_records\':len(ids),\n            \'unique_question_ids\':len(counts),\'duplicated_question_ids\':sum(n>1 for n in counts.values()),\n            \'conflicting_question_ids\':sum(len(v)>1 for v in variants.values()),\n            \'different_record_fields\':dict(differing_fields),\n            \'invalid_lines\':invalid,\'matching_two_halves\':bool(ids) and len(ids)%2==0 and ids[:halfway]==ids[halfway:]}\n\n\ndef evaluation_processes(root,proc_root=Path(\'/proc\')):\n    """Read only our own project workers; no process is killed."""\n    root=str(Path(root).resolve());found=[]\n    for path in Path(proc_root).glob(\'[0-9]*\'):\n        try:\n            pid=int(path.name)\n            if pid==os.getpid() or path.stat().st_uid!=os.getuid():continue\n            argv=[x.decode(\'utf-8\',errors=\'replace\') for x in (path/\'cmdline\').read_bytes().split(b\'\\0\') if x]\n            if not any(root in arg for arg in argv):continue\n            names={Path(arg).name for arg in argv}\n            is_evaluator=\'evaluate.py\' in names\n            is_runner=bool(names & {\'run_method_b.py\',\'run_method_b_eval_recovery.py\'})\n            is_evaluation_stage=\'evaluate\' in argv or (\'--action\' in argv and\n                    argv.index(\'--action\')+1<len(argv) and argv[argv.index(\'--action\')+1]==\'evaluate\')\n            if is_evaluator or (is_runner and is_evaluation_stage):\n                found.append({\'pid\':pid,\'program\':\'evaluate.py\' if is_evaluator else \'Method B evaluation runner\'})\n        except (OSError,ValueError):continue\n    return found\n\n\n@contextmanager\ndef single_evaluation(root):\n    folder=Path(root)/\'outputs\'/STUDY\n    folder.mkdir(parents=True,exist_ok=True)\n    path=folder/\'evaluation_recovery.lock\'\n    with path.open(\'a+\') as handle:\n        try:fcntl.flock(handle.fileno(),fcntl.LOCK_EX|fcntl.LOCK_NB)\n        except BlockingIOError:\n            raise RuntimeError(\'Another recovery evaluation is running. Leave it to finish; no files were changed.\')\n        active=evaluation_processes(root)\n        if active:\n            raise RuntimeError(\'Existing evaluation workers are still active: \'+json.dumps(active)+\n                \'. No files were changed. Wait for those workers to exit before starting recovery.\')\n        try:yield\n        finally:fcntl.flock(handle.fileno(),fcntl.LOCK_UN)\n\n\ndef archive_prediction_bundle(root,prediction,reason=\'Existing output lacks a matching verified completion record\'):\n    root=Path(root).resolve();base=root/\'outputs\'/STUDY\n    prediction=Path(prediction)\n    if prediction.is_symlink():raise ValueError(\'Refusing to move a symlink prediction: \'+str(prediction))\n    relative=prediction.resolve().relative_to(base.resolve())\n    if not (\'parent_evaluation_cache\' in relative.parts or \'evaluation\' in relative.parts):\n        raise ValueError(\'Recovery is restricted to evaluation outputs: \'+str(prediction))\n    if prediction.suffix!=\'.jsonl\':raise ValueError(\'Expected a prediction JSONL file\')\n    paths=[prediction,prediction.with_suffix(\'.config.json\'),prediction.with_suffix(\'.validated.json\')]\n    existing=[p for p in paths if p.exists()]\n    if not existing:return None\n    audit=inspect_predictions(prediction)\n    folder=base/\'evaluation_recovery_archives\'/(\n           datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%S%fZ\')+\'_\'+uuid.uuid4().hex[:8])/relative.parent\n    folder.mkdir(parents=True,exist_ok=False)\n    moved=[]\n    for path in existing:\n        if path.is_symlink():raise ValueError(\'Refusing symlink metadata: \'+str(path))\n        identity=sha(path)\n        target=folder/path.name\n        path.rename(target)\n        if sha(target)!=identity:raise RuntimeError(\'Archived file checksum differs: \'+str(target))\n        moved.append({\'original\':str(path),\'archived\':str(target),\'sha256\':identity})\n    report={\'reason\':reason,\'inspection\':audit,\'moved\':moved,\n            \'records_cherry_picked\':False,\'training_changed\':False,\n            \'next_action\':\'Regenerate this output using the unchanged pinned evaluator\'}\n    (folder/\'recovery_audit.json\').write_text(json.dumps(report,indent=2)+\'\\n\')\n    print(\'Archived previous evaluation output:\',folder,flush=True)\n    print(\'Regenerating this prediction file. No training is repeated.\',flush=True)\n    return report\n\n\ndef enter(main):\n    import sys\n    args=sys.argv[1:]\n    if not args:raise ValueError(\'Specify evaluate, summary, or export\')\n    if \'--root\' not in args:raise ValueError(\'Expected --root PROJECT\')\n    root=Path(args[args.index(\'--root\')+1])\n    if args[0]==\'evaluate\':\n        with single_evaluation(root):main()\n    elif args[0]==\'engine\' and \'--action\' in args and args[args.index(\'--action\')+1]==\'evaluate\':\n        main()\n    elif args[0] in (\'summary\',\'export\'):main()\n    else:raise ValueError(\'This recovery runner only permits evaluation, summary, and export\')\n', 'patch_runner.py': '"""Create an evaluation-only recovery copy; the original helper is not edited."""\nimport ast\n\n\ndef replace_once(source,old,new):\n    if source.count(old)!=1:raise ValueError(\'Source differs from reviewed evaluation runner: \'+old[:65])\n    return source.replace(old,new,1)\n\n\ndef patch_runner(source):\n    source=replace_once(source,"from select_b import (ORTOOLS_VERSION, VERSION, bounds, cell, csv_bytes, digest,",\n                        "from evaluation_recovery import archive_prediction_bundle, enter\\n\\nfrom select_b import (ORTOOLS_VERSION, VERSION, bounds, cell, csv_bytes, digest,")\n    old="""                if pred.exists() or marker.exists():\n                    raise FileExistsError(\'Partial/different predictions were preserved: \'+str(pred)+\n                                          \'. Inspect them before starting a separately named retry.\')\n"""\n    new="""                if pred.exists() or marker.exists() or pred.with_suffix(\'.config.json\').exists():\n                    archive_prediction_bundle(root,pred)\n"""\n    source=replace_once(source,old,new)\n    old="""            if valid:\n                validate_pair(pred,pred)\n"""\n    new="""            if valid:\n                try:\n                    saved_runtime=read(pred.with_suffix(\'.config.json\'))\n                    valid=(saved_runtime.get(\'gpu\')==torch.cuda.get_device_name(0) and\n                           saved_runtime.get(\'torch\')==torch.__version__ and\n                           saved_runtime.get(\'cuda\')==torch.version.cuda)\n                except (OSError,ValueError):\n                    valid=False\n            if valid:\n                validate_pair(pred,pred)\n"""\n    source=replace_once(source,old,new)\n    old="""            if marker.is_file() and pred.is_file():\n                prior=read(marker)\n                valid=prior.get(\'signature\')==sig and prior.get(\'predictions_sha256\')==digest(pred) and prior.get(\'config_sha256\')==digest(pred.with_suffix(\'.config.json\'))\n"""\n    new="""            if marker.is_file() and pred.is_file():\n                try:\n                    prior=read(marker)\n                    valid=prior.get(\'signature\')==sig and prior.get(\'predictions_sha256\')==digest(pred) and prior.get(\'config_sha256\')==digest(pred.with_suffix(\'.config.json\'))\n                except (OSError,ValueError,KeyError):\n                    valid=False\n"""\n    source=replace_once(source,old,new)\n    if source.count("HERE/\'run_method_b.py\'")!=2:\n        raise ValueError(\'Unexpected number of subprocess script references\')\n    source=source.replace("HERE/\'run_method_b.py\'","Path(__file__).resolve()")\n    source=replace_once(source,"if __name__==\'__main__\':main()","if __name__==\'__main__\':enter(main)")\n    ast.parse(source)\n    return source\n'}
EXPECTED_ORIGINAL_SHA256 = 'dc743c10ca08f1df00cfb89d5b80a6ce8d1571a6693e2ad0a4762bb587869b06'

from pathlib import Path
import ast, hashlib, importlib.util, json, subprocess, sys

ROOT = Path('/scratch/sshind38/capstone_approach_b')
CODE = ROOT / 'method_b_document_v3_code'
PYTHON = ROOT / 'venv_b_training_py312/bin/python'
ORIGINAL = CODE / 'run_method_b.py'
assert PYTHON.is_file(), f'Missing training environment: {PYTHON}'
assert ORIGINAL.is_file(), f'Missing existing helper: {ORIGINAL}'
assert hashlib.sha256(ORIGINAL.read_bytes()).hexdigest() == EXPECTED_ORIGINAL_SHA256, (
    'The original helper differs from the reviewed version. It was preserved; share the changed file before adapting it.')

def install_file(path, data):
    if path.exists() and path.read_bytes() != data:
        raise FileExistsError(f'Existing recovery version differs and was preserved: {path}')
    path.write_bytes(data)

for name, content in PAYLOAD.items():
    ast.parse(content)
    install_file(CODE / name, content.encode())

spec = importlib.util.spec_from_file_location('method_b_recovery_patcher', CODE/'patch_runner.py')
patcher = importlib.util.module_from_spec(spec)
spec.loader.exec_module(patcher)
RECOVERY = CODE / 'run_method_b_eval_recovery.py'
patched = patcher.patch_runner(ORIGINAL.read_text())
install_file(RECOVERY, patched.encode())
print('Recovery helper ready:', RECOVERY)
print('Original helper unchanged:', hashlib.sha256(ORIGINAL.read_bytes()).hexdigest() == EXPECTED_ORIGINAL_SHA256)


## 2. Check the affected file and confirm there are no active evaluation workers

This is a read-only audit. If it reports active workers, leave this notebook idle until those workers
have exited. Starting a second evaluation while one is active could append into the same outputs.
The duplicate counts alone do not establish whether overlapping writers or a later append caused the error.


In [ ]:
audit_code = r"""
from pathlib import Path
import json, sys
root=Path('/scratch/sshind38/capstone_approach_b')
sys.path.insert(0,str(root/'method_b_document_v3_code'))
from evaluation_recovery import evaluation_processes,inspect_predictions
workers=evaluation_processes(root)
print('Existing evaluation workers:',workers)
p=root/'outputs/B_method_B_document_v3/parent_evaluation_cache/0.5B_seed123_dev_gsm_48fab29f1001e906703e.jsonl'
print(json.dumps(inspect_predictions(p),indent=2))
if workers:raise SystemExit('Evaluation workers are still active. No predictions were changed. Wait before starting recovery.')
"""
subprocess.run([str(PYTHON), '-u', '-c', audit_code], check=True)


## 3. Recover the failed output and continue the primary evaluations

This requires the GPU session. It checks for a single visible NVIDIA GPU and the completed training
audits, reuses matching validated results, archives unvalidated output bundles, and runs the remaining
evaluations. The same questions, precision, decoding, parser, and adapter identities are retained.

Backups are saved under:

`outputs/B_method_B_document_v3/evaluation_recovery_archives/`

Each backup has an audit containing the original hashes and duplicate counts. No record is chosen
according to its correctness. If a newly generated file again fails validation, the process stops and
keeps its output for inspection; it does not repeatedly rerun in a loop.


In [ ]:
from datetime import datetime, timezone

def recover_stage(stage):
    assert stage in ('evaluate', 'summary', 'export')
    command = [str(PYTHON), '-u', str(RECOVERY), stage,
               '--root', str(ROOT), '--setting', 'mixed', '--budget', '1',
               '--parent-seeds', '17', '123', '--training-seeds', '42']
    logs = ROOT / 'outputs/B_method_B_document_v3/logs'
    logs.mkdir(parents=True, exist_ok=True)
    name = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    log = logs / f'evaluation_recovery_{stage}_{name}.log'
    print('Stage:', stage, '| log:', log, flush=True)
    with log.open('w', encoding='utf-8') as out:
        with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1, cwd=str(ROOT)) as process:
            for line in process.stdout:
                print(line, end='', flush=True)
                out.write(line)
                out.flush()
            code = process.wait()
    if code:
        raise RuntimeError(f'{stage} stopped (exit {code}). Read the error above. Log: {log}')
    print('FINISHED:', stage, flush=True)

recover_stage('evaluate')


## 4. Save the summary and handoff archive

Run this after stage 3 finishes successfully. It writes the primary development comparisons and
packages the final adapters, selected data, logs, code, and predictions. No additional budgets,
confirmation-seed training, transfer training, or final-test evaluation are launched.


In [ ]:
recover_stage('summary')
recover_stage('export')


**Validation:** 16 local recovery tests cover conflicting records, preservation of original bytes,
scope restrictions, worker detection, locking, and the evaluation-only entry point. The actual
regeneration and GPU evaluation run on Sol against your existing artifacts.
